# Batch Whisper Transcription — Kaggle GPU

## 1. Install

In [ ]:
!pip install faster-whisper -q
!pip install demucs -q

## 2. Config

In [ ]:
import os
from pathlib import Path

# ── Thư mục chứa videos ──
VIDEO_DIR   = "./video"
VIDEO_EXT   = ".mp4"

# ── Output ──
OUTPUT_DIR  = "./transcripts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model config ──
MODEL_SIZE   = "large-v3"
LANGUAGE     = "vi"
COMPUTE_TYPE = "float16"   # float16 cho GPU, int8 nếu CPU

MAX_RUNTIME_HOURS = 8.0

# ── Segment grouping config ──
SIM_THRESHOLD = 0.40
MAX_GAP_S     = 3.0

DEMUCS_MODEL = "mdx_extra"  
#htdemucs_ft" ← fine-tuned version, tốt hơn htdemucs
#mdx_extra" ← tốt nhất cho vocal separation
DEMUCS_DIR = "separated"

HALLUCINATION_PATTERNS = [
    # Subscribe / đăng ký + kênh
    r"(hãy |nhớ |đừng quên )(subscribe|đăng ký)( cho)? kênh",

    # Like — bắt buộc có động từ kêu gọi đứng trước
    r"(nhấn |hãy |đừng quên |bấm )like( video)?",

    # Chuông thông báo
    r"(nhấn |bật |click )?(vào )?chuông thông báo",

    # Share đơn — bắt buộc có từ kêu gọi đứng trước
    r"(hãy |nhớ |đừng quên |nhấn |bấm )(share|chia sẻ) (video|kênh)",

    # Combo like...share — bắt buộc có từ kêu gọi đứng trước
    r"(nhấn |hãy |đừng quên |bấm |nhớ )like.{0,20}share",
    r"(nhấn |hãy |đừng quên |bấm |nhớ )(like|share).{0,20}(đăng ký|subscribe)",
    r"(nhấn |hãy |đừng quên |bấm |nhớ )(đăng ký|subscribe).{0,20}(like|share)",

    # Combo bấm chuông + chia sẻ
    r"bấm chuông.{0,20}chia sẻ",
]

print(f"Video dir : {VIDEO_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Model     : {MODEL_SIZE} | lang={LANGUAGE} | compute={COMPUTE_TYPE}")
print(f"Max runtime: {MAX_RUNTIME_HOURS}h")
print(f"Demucs model: {DEMUCS_MODEL}")

## 3. GPU Check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 4. Scan videos & kiểm tra checkpoint

In [ ]:
#chạy hết
BATCH_INDEX = None  

#chạy theo phần
BATCH_TOTAL = 2
#BATCH_INDEX = 0
#BATCH_INDEX = 1

In [ ]:
all_videos = sorted(Path(VIDEO_DIR).glob(f"*{VIDEO_EXT}"))

def get_output_path(video_path):
    return Path(OUTPUT_DIR) / f"{video_path.stem}_transcript.json"

# Chia batch
if BATCH_INDEX is not None:
    half = len(all_videos) // 2
    all_videos = all_videos[:half] if BATCH_INDEX == 0 else all_videos[half:]

done    = [v for v in all_videos if get_output_path(v).exists()]
pending = [v for v in all_videos if not get_output_path(v).exists()]

print(f"Total videos : {len(all_videos)}")
print(f"Already done : {len(done)}")
print(f"Pending      : {len(pending)}")
print()
if done:
    print("Done (skip):")
    for v in done:
        print(f"  ✅ {v.name}")
print("\nPending:")
for v in pending:
    print(f"  ⏳ {v.name}")

## 5. Load model

In [6]:
import time
from faster_whisper import WhisperModel

t0 = time.time()
model = WhisperModel(MODEL_SIZE, device="cuda", 
                     compute_type=COMPUTE_TYPE)
print(f"Model loaded in {time.time()-t0:.3f}s")

Model loaded in 19.063s


## 6. Helper functions

In [ ]:
import json
import numpy as np
import subprocess
import shutil
import re

def separate_vocals(video_path):
    t = time.time()
    result = subprocess.run([
        "python", "-m", "demucs",
        "--two-stems=vocals",
        "-n", DEMUCS_MODEL,
        "--device", "cuda",
        "--out", DEMUCS_DIR,
        str(video_path)
    ], capture_output=True, text=True)  # ← tắt output, bắt stderr
    if result.returncode != 0:
        raise RuntimeError(f"Demucs failed: {result.stderr[-300:]}")
    print(f"   🎵 Demucs: {time.time()-t:.1f}s")
    return Path(DEMUCS_DIR) / DEMUCS_MODEL / Path(video_path).stem / "vocals.wav"

def is_hallucination(seg) -> bool:
    text = seg.text.lower().strip()
    if any(re.search(pat, text) for pat in HALLUCINATION_PATTERNS):
        print(f"   🚫 Hallucination: {seg.text.strip()}")
        return True
    return False

def transcribe_video(video_path):
    vocal_path = separate_vocals(video_path)
    try:
        t_start = time.time()
        segments_gen, info = model.transcribe(
            str(vocal_path),
            language=LANGUAGE,
            word_timestamps=True,
            vad_filter=True,
            vad_parameters={
                "min_silence_duration_ms": 500, 
                "speech_pad_ms": 200},
            beam_size=5,
            temperature=0.0,
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
            log_prob_threshold=-1.0,
            compression_ratio_threshold=2.4,
        )
        segments = [
            s for s in list(segments_gen)
            if s.no_speech_prob < 0.6 and not is_hallucination(s)
        ]
        raw_segments = []
        for i, seg in enumerate(segments):
            raw_segments.append({
                "id"   : i,
                "start": round(seg.start, 3),
                "end"  : round(seg.end, 3),
                "text" : seg.text.strip(),
                "words": [
                    {"word": w.word, "start": round(w.start, 3),
                     "end": round(w.end, 3), "prob": round(w.probability, 4)}
                    for w in (seg.words or [])
                ],
            })
        elapsed = time.time() - t_start
        rtf = elapsed / info.duration if info.duration > 0 else 0
        return {
            "video"   : video_path.stem,
            "language": info.language,
            "duration": round(info.duration, 4),
            "rtf"     : round(rtf, 6),
            "model"   : MODEL_SIZE,
            "segments": raw_segments,
        }, elapsed
    finally:
        shutil.rmtree(
            Path(DEMUCS_DIR) / DEMUCS_MODEL / Path(video_path).stem,
            ignore_errors=True
        )

print("Helper functions loaded ✅")

## 7. Batch loop — tự động resume & dừng trước khi hết giờ

In [ ]:
SESSION_START  = time.time()
MAX_RUNTIME_S  = MAX_RUNTIME_HOURS * 3600

stats = {"ok": [], "error": [], "skipped": len(done)}

def time_left():
    return MAX_RUNTIME_S - (time.time() - SESSION_START)

def fmt_time(seconds):
    h, m = divmod(int(seconds), 3600)
    m, s = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

print(f"Starting batch — {len(pending)} videos to process")
print(f"Time budget: {fmt_time(MAX_RUNTIME_S)}")
print("=" * 70)

for idx, video_path in enumerate(pending):
    remaining = time_left()

    if remaining < 600:
        print(f"\n⏰ Only {fmt_time(remaining)} left — stopping to avoid session kill.")
        print(f"   Resume next session: {len(pending) - idx} videos left.")
        break

    out_path = get_output_path(video_path)

    # Double-check
    if out_path.exists():
        print(f"[{idx+1}/{len(pending)}] ✅ Skip (already done): {video_path.name}")
        stats["skipped"] += 1
        continue

    print(f"\n[{idx+1}/{len(pending)}] 🎬 {video_path.name}  "
          f"(time left: {fmt_time(remaining)})")

    try:
        result, elapsed = transcribe_video(video_path)

        # Save ngay sau khi xong → không mất nếu crash sau này
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        stats["ok"].append(video_path.name)
        print(f"   ✅ Done  {elapsed:.1f}s  RTF={result['rtf']:.3f}  "
              f"segs={len(result['segments'])} "
              f"→ {out_path.stat().st_size/1024:.1f} KB")

    except Exception as e:
        stats["error"].append((video_path.name, str(e)))
        print(f"   ❌ ERROR: {e}")
        # Không dừng — tiếp tục video tiếp theo
        continue

print("\n" + "=" * 70)
print("SESSION SUMMARY")
print(f"  ✅ Success : {len(stats['ok'])}")
print(f"  ⏭️  Skipped : {stats['skipped']}")
print(f"  ❌ Errors  : {len(stats['error'])}")
print(f"  ⏱️  Runtime : {fmt_time(time.time() - SESSION_START)}")
if stats["error"]:
    print("\nFailed videos:")
    for name, err in stats["error"]:
        print(f"  {name}: {err}")

## 8. Gộp tất cả JSON thành 1 file tổng (chạy sau khi xong hết)

In [ ]:
all_results = []
missing     = []

for video_path in all_videos:
    out_path = get_output_path(video_path)
    if out_path.exists():
        with open(out_path, encoding="utf-8") as f:
            all_results.append(json.load(f))
    else:
        missing.append(video_path.name)

merged_path = Path(OUTPUT_DIR) / "all_transcripts.json"
with open(merged_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"Merged {len(all_results)} transcripts → {merged_path}")
print(f"File size: {merged_path.stat().st_size / 1024 / 1024:.2f} MB")
if missing:
    print(f"\n⚠️  {len(missing)} videos chưa có transcript:")
    for m in missing:
        print(f"  {m}")

## 9. Quick quality check trên toàn bộ

In [ ]:
import statistics

total_dur_s = sum(r["duration"] for r in all_results)
rtfs        = [r["rtf"] for r in all_results]
seg_counts  = [len(r["segments"]) for r in all_results]

print("=" * 60)
print(f"Videos processed : {len(all_results)}")
print(f"Total audio      : {total_dur_s/3600:.2f}h")
print(f"RTF avg/med      : {statistics.mean(rtfs):.3f} / {statistics.median(rtfs):.3f}")
print(f"Segments avg     : {statistics.mean(seg_counts):.1f}")
print()

# Videos có RTF bất thường cao (chạy chậm → có thể lỗi model)
slow = [(r["video"], r["rtf"]) for r in all_results if r["rtf"] > 0.5]
if slow:
    print(f"⚠️  Videos có RTF > 0.5 (chậm bất thường):")
    for name, rtf in slow:
        print(f"   {name}: RTF={rtf:.3f}")

# Words có confidence thấp
LOW_PROB = 0.6
total_low = sum(
    sum(1 for seg in r["segments"] for w in seg["words"] if w["prob"] < LOW_PROB)
    for r in all_results
)
total_words = sum(
    sum(len(seg["words"]) for seg in r["segments"])
    for r in all_results
)
print(f"\nLow-confidence words (prob<{LOW_PROB}): {total_low}/{total_words} "
      f"({100*total_low/total_words:.1f}%)" if total_words else "no words")